In [1]:
# ================================
# IMPORT REQUIRED LIBRARIES
# ================================

import os
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from collections import Counter
from pathlib import Path


In [2]:
# ================================
# DEVICE CONFIGURATION
# ================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cpu


In [3]:
# ================================
# DEFINE PROJECT PATHS
# ================================

PROJECT_ROOT = Path().resolve().parent

PROCESSED_PATH = PROJECT_ROOT / "data" / "processed"
TRAIN_PATH = PROCESSED_PATH / "train"
VAL_PATH = PROCESSED_PATH / "val"
TEST_PATH = PROCESSED_PATH / "test"

print("Train Path:", TRAIN_PATH)
print("Validation Path:", VAL_PATH)
print("Test Path:", TEST_PATH)


Train Path: C:\Users\anish\OneDrive\Desktop\plantvillage\data\processed\train
Validation Path: C:\Users\anish\OneDrive\Desktop\plantvillage\data\processed\val
Test Path: C:\Users\anish\OneDrive\Desktop\plantvillage\data\processed\test


In [4]:
# ================================
# DEFINE TRANSFORM PIPELINES
# ================================

# Train Transforms (With Augmentation)
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Validation & Test Transforms (No Augmentation)
val_test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


In [5]:
# ================================
# LOAD DATASETS
# ================================

train_dataset = datasets.ImageFolder(TRAIN_PATH, transform=train_transforms)
val_dataset = datasets.ImageFolder(VAL_PATH, transform=val_test_transforms)
test_dataset = datasets.ImageFolder(TEST_PATH, transform=val_test_transforms)

print("Number of Classes:", len(train_dataset.classes))
print("Classes:", train_dataset.classes)


Number of Classes: 38
Classes: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___healthy', 'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew', 'Strawberry___Leaf_scorch', 'Strawberry___healthy', 'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot', 'Tomato___Spider_mites Two-spotted_sp

In [6]:
# ================================
# VERIFY CLASS STRUCTURE
# ================================

if train_dataset.classes == val_dataset.classes:
    print("Train and validation datasets have identical class structure.")
else:
    print("Warning: Train and validation classes differ!")


Train and validation datasets have identical class structure.


In [7]:
# ================================
# CREATE DATALOADERS
# ================================

BATCH_SIZE = 32
NUM_WORKERS = 2

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print("Train Batches:", len(train_loader))
print("Validation Batches:", len(val_loader))
print("Test Batches:", len(test_loader))


Train Batches: 1188
Validation Batches: 255
Test Batches: 256


In [8]:
# ================================
# HANDLE CLASS IMBALANCE
# ================================

# Extract labels from training dataset
train_labels = [label for _, label in train_dataset]

# Count samples per class
class_counts = Counter(train_labels)

# Total samples
total_samples = sum(class_counts.values())

# Compute class weights
class_weights = [
    total_samples / (len(class_counts) * class_counts[i])
    for i in range(len(class_counts))
]

class_weights = torch.tensor(class_weights, dtype=torch.float)

print("Class Weights:", class_weights)

# Define weighted loss function
criterion = torch.nn.CrossEntropyLoss(weight=class_weights.to(device))


Class Weights: tensor([2.2674, 2.3040, 5.2079, 0.8687, 0.9514, 1.3586, 1.6749, 2.7853, 1.1989,
        1.4513, 1.2299, 1.2106, 1.0330, 1.3279, 3.3781, 0.2595, 0.6222, 3.9837,
        1.4346, 0.9670, 1.4285, 1.4285, 9.4332, 3.8607, 0.2806, 0.7788, 1.2886,
        3.1345, 0.6720, 1.4285, 0.7484, 1.5014, 0.8070, 0.8524, 1.0182, 0.2667,
        3.8311, 0.8984])


In [9]:
# ================================
# VERIFY PIPELINE
# ================================

images, labels = next(iter(train_loader))

print("Image Batch Shape:", images.shape)
print("Label Batch Shape:", labels.shape)
print("Image Data Type:", images.dtype)
print("Value Range:", images.min().item(), images.max().item())


c:\Users\anish\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Image Batch Shape: torch.Size([32, 3, 224, 224])
Label Batch Shape: torch.Size([32])
Image Data Type: torch.float32
Value Range: -2.1179039478302 2.640000104904175
